# Hand Action Detection Notebook

This notebook is part of a pipeline for hand pose sequence capture, dataset creation, model training, and inference. Comments and docs are concise and in English.


In [2]:
import cv2
import numpy as np
import time
import json
import queue
from typing import List, Optional, Literal, Tuple
from tensorflow.keras.models import Model, load_model
from landmarkers.inferences import InferenceSequence, Inference
from landmarkers.landmarks import LandmarksSequence
from landmarkers.mp.hands import MediapipeHandsMetadata, MPLiveStreamLandmarker


def load_config(config_path: str):
    """
    Load configuration from JSON file.

    Args:
        config_path: Path to config file.

    Returns:
        Parsed configuration dictionary.
    """
    with open(config_path, 'r') as f:
        return json.load(f)


def get_timestamp_ms() -> int:
    """
    Get current timestamp in milliseconds.

    Returns:
        Current time in milliseconds.
    """
    return int(time.time() * 1000)


def sigmoid(x: np.ndarray) -> np.ndarray:
    """
    Numerically stable sigmoid function.

    Args:
        x: Input array.

    Returns:
        Sigmoid-transformed array.
    """
    return np.where(
        x >= 0,
        1 / (1 + np.exp(-x)),
        np.exp(x) / (1 + np.exp(x))
    )


def get_centroid_velocity_norm_sequence(landmark_sequence) -> np.ndarray:
    """
    Compute normalized centroid velocity using a sigmoid scaling.

    Args:
        landmark_sequence: LandmarksSequence object.

    Returns:
        Array of shape (T, 3) representing normalized velocity.
    """
    time_stamps = np.array(landmark_sequence.time_stamps_ms, dtype=np.float32)
    centroids = np.array(landmark_sequence.centroid(), dtype=np.float32)

    d_centroids = np.diff(centroids, axis=0)
    d_time = np.diff(time_stamps)[:, None]
    d_time[d_time == 0] = 1e-6

    velocity = d_centroids / d_time

    norms = np.linalg.norm(velocity, axis=1, keepdims=True)
    norms[norms == 0] = 1e-6

    norm_velocity = (sigmoid(norms) / norms) * velocity

    norm_velocity = np.vstack([
        np.zeros((1, 3), dtype=np.float32),
        norm_velocity
    ])

    return norm_velocity.astype(np.float32)


def preprocess_sequence(
    right_seq: InferenceSequence,
    left_seq: InferenceSequence,
    sequence_length: int,
    hand_selection: str
) -> np.ndarray:
    """
    Convert raw inference sequences into model-ready features.

    Args:
        right_seq: Right hand sequence.
        left_seq: Left hand sequence.
        sequence_length: Target sequence length.
        hand_selection: 'left', 'right', or 'both'.

    Returns:
        Feature array of shape (sequence_length, features).
    """
    def process_hand(seq):
        landmark_seq: LandmarksSequence = seq.landmarks_sequence
        landmark_seq = landmark_seq.resample()

        velocity = get_centroid_velocity_norm_sequence(landmark_seq)
        landmark_seq = landmark_seq.centered(0)

        arr = landmark_seq.array
        return np.concatenate([arr, velocity[:, None, :]], axis=1)

    r_array = process_hand(right_seq)
    l_array = process_hand(left_seq)

    n_frames = sequence_length

    if r_array.shape[0] != n_frames:
        idxs = np.linspace(0, r_array.shape[0] - 1, n_frames, dtype=int)
        r_array = r_array[idxs]

    if l_array.shape[0] != n_frames:
        idxs = np.linspace(0, l_array.shape[0] - 1, n_frames, dtype=int)
        l_array = l_array[idxs]

    frames = []

    for i in range(n_frames):
        if hand_selection == 'left':
            frames.append(l_array[i].reshape(-1))
        elif hand_selection == 'right':
            frames.append(r_array[i].reshape(-1))
        else:
            frames.append(np.hstack([
                r_array[i].reshape(-1),
                l_array[i].reshape(-1)
            ]))

    return np.array(frames, dtype=np.float32)


def predict_gesture(
    features: np.ndarray,
    model: Model,
    pred_buffer: List[np.ndarray],
    actions: np.ndarray,
    threshold: float,
    buffer_size: int
) -> Optional[str]:
    """
    Predict gesture using temporal smoothing.

    Args:
        features: Input feature sequence.
        model: Trained model.
        pred_buffer: Buffer of past predictions.
        actions: Label names.
        threshold: Confidence threshold.
        buffer_size: Number of predictions for smoothing.

    Returns:
        Predicted gesture label or None.
    """
    features = np.expand_dims(features, axis=0)
    res = model.predict(features, verbose=0)[0]

    pred_buffer.append(res)
    avg = np.mean(pred_buffer[-buffer_size:], axis=0)

    idx = int(np.argmax(avg))

    if avg[idx] > threshold and actions[idx] != "none":
        return str(actions[idx])

    return None


def get_hand(
    inferences: Optional[List[Inference]],
    category: Literal["Left", "Right"],
    num_landmarks: int
) -> Inference:
    """
    Retrieve hand inference or return an empty placeholder.

    Args:
        inferences: List of detected hands.
        category: Hand type ('Left' or 'Right').
        num_landmarks: Number of landmarks.

    Returns:
        Inference object.
    """
    if inferences is None:
        zeros = np.zeros((num_landmarks, 3), dtype=np.float32)
        return Inference(zeros, zeros, MediapipeHandsMetadata(category_name=category, index=0, score=0.0))

    hands = [h for h in inferences if h.metadata.category_name == category]

    if hands:
        return hands[0]

    zeros = np.zeros((num_landmarks, 3), dtype=np.float32)
    return Inference(zeros, zeros, MediapipeHandsMetadata(category_name=category, index=0, score=0.0))


def render_frame(
    frame: np.ndarray,
    right_hand: Inference,
    left_hand: Inference,
    gesture: Optional[str]
) -> np.ndarray:
    """
    Draw hand landmarks and predicted gesture on frame.

    Args:
        frame: Input image.
        right_hand: Right hand inference.
        left_hand: Left hand inference.
        gesture: Predicted gesture.

    Returns:
        Annotated frame.
    """
    for lm in right_hand.landmarks.array:
        x, y = int(lm[0] * frame.shape[1]), int(lm[1] * frame.shape[0])
        cv2.circle(frame, (x, y), 3, (0, 255, 0), -1)

    for lm in left_hand.landmarks.array:
        x, y = int(lm[0] * frame.shape[1]), int(lm[1] * frame.shape[0])
        cv2.circle(frame, (x, y), 3, (0, 0, 255), -1)

    if gesture:
        cv2.putText(frame, gesture, (50, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3)

    return frame


def create_stream_callback(frame_queue):
    """
    Create callback function for live stream inference.

    Args:
        frame_queue: Queue to store frames.

    Returns:
        Callback function.
    """
    def callback(inferences, frame, timestamp_ms):
        try:
            frame_queue.put_nowait((frame, inferences, timestamp_ms))
        except queue.Full:
            pass

    return callback


def main():
    config = load_config('../config.json')

    common_config = config['common']
    ACTIONS = np.array(common_config['actions'])
    SEQUENCE_LENGTH = common_config['sequence_length']

    train_config = config['train_model']
    MODEL_EXPORT_NAME = train_config['model_export_name']
    HAND_SELECTION = train_config['hand_selection']

    THRESHOLD = 0.85
    PRED_BUFFER_SIZE = 10
    NUM_LANDMARKS = 21

    model: Model = load_model(MODEL_EXPORT_NAME)

    seq_right = InferenceSequence(fixed_buffer_length=SEQUENCE_LENGTH)
    seq_left = InferenceSequence(fixed_buffer_length=SEQUENCE_LENGTH)
    pred_buffer: List[np.ndarray] = []

    frame_queue: "queue.Queue[Tuple[np.ndarray, List[Inference], int]]" = queue.Queue(maxsize=1)

    cap = cv2.VideoCapture(0)

    callback = create_stream_callback(frame_queue)

    with MPLiveStreamLandmarker(
        model_path="hand_landmarker.task",
        callback=callback,
        num_hands=2
    ) as live_landmarker:

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            ts = get_timestamp_ms()
            live_landmarker.infer(frame, ts)

            if not frame_queue.empty():
                f, inferences, ts = frame_queue.get()

                right_hand = get_hand(inferences, "Right", NUM_LANDMARKS)
                left_hand = get_hand(inferences, "Left", NUM_LANDMARKS)

                seq_right.append(right_hand, ts)
                seq_left.append(left_hand, ts)

                gesture = None

                if len(seq_right) == SEQUENCE_LENGTH and len(seq_left) == SEQUENCE_LENGTH:
                    features = preprocess_sequence(
                        seq_right,
                        seq_left,
                        SEQUENCE_LENGTH,
                        HAND_SELECTION
                    )

                    gesture = predict_gesture(
                        features,
                        model,
                        pred_buffer,
                        ACTIONS,
                        THRESHOLD,
                        PRED_BUFFER_SIZE
                    )

                show_frame = render_frame(f, right_hand, left_hand, gesture)
                cv2.imshow("Feed", show_frame)

            if cv2.waitKey(10) & 0xFF == ord("q"):
                break

    cap.release()
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()

2026-04-04 15:06:45.240006: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1775308005.515172  824549 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775308005.529158  824549 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775308005.859525  824554 landmark_projection_calculator.cc:78] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
/tmp/ipykernel_824146/1040893442.py:50: RuntimeWarning: overflow encountered in exp
  np.exp(x) / (1 + np.exp(x))
/tmp/ipykernel_824146/1040893442.py:50: RuntimeWarning: invalid value encount